<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/CenterPointHead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np

from google.colab import drive
#drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
CUDA_version = torch.version.cuda.replace('.', '')

!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

import torch_sparse

from scipy.ndimage import maximum_filter

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 130.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 96.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 38.7 MB/s eta 0:00:00


In [ ]:
def generate_heatmap(w,h, centers,sigma):

  x_axis = torch.arange(0, w, device=centers.device, dtype=torch.float32)
  y_axis = torch.arange(0, h, device=centers.device, dtype=torch.float32)
  y_grid, x_grid = torch.meshgrid(y_axis, x_axis, indexing='ij')

  distances_squared = torch.zeros((w,h), device=centers.device)

  for i in range(centers.shape[0]):
    distances_squared += (x_grid-centers[i][0])**2 +(y_grid-centers[i][1])

  heatmap = torch.exp(-1*distances_squared/(2*sigma**2))

  return heatmap

In [ ]:
class CenterPoint(nn.Module):

  def __init__(self, w, h, C, num_classes):
    super().__init__()

    self.w = w
    self.h = h
    self.num_classes = num_classes
    self.C = C

    self.heat_conv1 = nn.Conv2d(self.C, self.C, kernel_size = 3, padding = 1)
    self.heatbn = nn.BatchNorm2d(C)
    self.heat_conv2 = nn.Conv2d(self.C, num_classes, kernel_size = 1)

  def learn_heatmap(self, feature_map):

    heatmap = F.relu(self.heatbn(self.heat_conv1(feature_map)))
    heatmap = F.sigmoid(self.conv2(heatmap))

    return heatmap

  def get_centers(self, heatmap):
    centers = torch.zeros((3))

    for k in range(self.num_classes):
      heatmap_k = heatmap[:,:, k]
      local_max_filter = maximum_filter(heatmap_k, size=3, mode='constant', cval=-np.inf)

      is_local_max = (heatmap_k == local_max_filter)

      rows, cols = np.where(is_local_max)






In [ ]:
test = CenterPoint(10,10, 300, 4)

In [ ]:
test = torch.zeros((2,3,4))

test[:,:,1]


tensor([[0., 0., 0.],
        [0., 0., 0.]])